In [ ]:
# Cell 1: Load and prepare GSE data
import pandas as pd
import numpy as np
import os
import pickle
from datetime import datetime
from biolearn.data_library import DataLibrary

def _log(message: str) -> None:
    """Log a message with timestamp."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")

def load_gse_data(accession: str):
    """
    Load GSE data using biolearn and save locally as CSV files.
    
    Args:
        accession (str): GSE accession number (e.g., 'GSE42861')
    
    Returns:
        tuple: (dnam_df, metadata_df) - methylation data and metadata as pandas DataFrames
    """
    _log(f"Starting {accession} data loading...")
    
    # Create local data directory
    data_dir = "2_poc_simulacra"
    os.makedirs(data_dir, exist_ok=True)
    
    # Check if data already exists
    dnam_path = os.path.join(data_dir, f"{accession}_dnam.csv")
    metadata_path = os.path.join(data_dir, f"{accession}_metadata.csv")
    
    if os.path.exists(dnam_path) and os.path.exists(metadata_path):
        _log(f"Loading existing {accession} data from local files...")
        dnam_df = pd.read_csv(dnam_path, index_col=0)
        metadata_df = pd.read_csv(metadata_path, index_col=0)
        _log(f"Loaded {accession}_dnam.csv: {dnam_df.shape[0]} rows, {dnam_df.shape[1]} columns")
        _log(f"Loaded {accession}_metadata.csv: {metadata_df.shape[0]} rows, {metadata_df.shape[1]} columns")
        return dnam_df, metadata_df
    
    _log(f"Downloading {accession} data from biolearn (LONG OPERATION)...")
    start_time = datetime.now()
    
    # Load data using biolearn
    library = DataLibrary()
    data = library.get(accession)
    if data is None:
        raise ValueError(f"{accession} dataset not found in biolearn library")
    
    data = data.load()
    
    # Extract methylation data (transpose to have samples as rows)
    dnam_df = data.dnam.T
    metadata_df = data.metadata.copy()
    
    # Ensure index alignment
    common_samples = dnam_df.index.intersection(metadata_df.index)
    dnam_df = dnam_df.loc[common_samples]
    metadata_df = metadata_df.loc[common_samples]
    
    download_duration = datetime.now() - start_time
    _log(f"Data download completed in {download_duration.total_seconds():.2f} seconds")
    
    _log(f"Saving {accession} data to local CSV files (LONG OPERATION)...")
    save_start = datetime.now()
    
    # Save methylation data
    dnam_df.to_csv(dnam_path)
    _log(f"Saved {accession}_dnam.csv: {dnam_df.shape[0]} rows, {dnam_df.shape[1]} columns")
    
    # Save metadata
    metadata_df.to_csv(metadata_path)
    _log(f"Saved {accession}_metadata.csv: {metadata_df.shape[0]} rows, {metadata_df.shape[1]} columns")
    
    save_duration = datetime.now() - save_start
    _log(f"Data saving completed in {save_duration.total_seconds():.2f} seconds")
    
    return dnam_df, metadata_df

# Execute the function for GSE42861
dnam_df, metadata_df = load_gse_data('GSE42861')

_log("GSE42861 data loading completed successfully!")
_log(f"Final dnam shape: {dnam_df.shape}")
_log(f"Final metadata shape: {metadata_df.shape}")
_log(f"Metadata columns: {list(metadata_df.columns)}")


[2025-10-12 12:30:34] Starting GSE42861 data loading...
[2025-10-12 12:30:34] Downloading GSE42861 data from biolearn (LONG OPERATION)...
[2025-10-12 12:30:49] Data download completed in 14.56 seconds
[2025-10-12 12:30:49] Saving data to local CSV files (LONG OPERATION)...
[2025-10-12 12:48:22] Saved dnam.csv: 689 rows, 485577 columns
[2025-10-12 12:48:22] Saved metadata.csv: 689 rows, 4 columns
[2025-10-12 12:48:22] Data saving completed in 1052.84 seconds
[2025-10-12 12:48:22] GSE42861 data loading completed successfully!
[2025-10-12 12:48:22] Final dnam shape: (689, 485577)
[2025-10-12 12:48:22] Final metadata shape: (689, 4)
[2025-10-12 12:48:22] Metadata columns: ['disease', 'age', 'sex', 'smoking']


In [ ]:
# Cell 2: Generate embeddings with Pythae
import os
import sys
import torch
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
from pythae.models import VAE, VAEConfig
from pythae.trainers import BaseTrainerConfig
from pythae.pipelines.training import TrainingPipeline
import glob

# Import our custom dataset classes
from iterable_csv_dataset import PythaeIterableDataset, DNAmDataset

def _log(message: str) -> None:
    """Log a message with timestamp."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")

def find_model_path(model_dir):
    """Find the actual model path within the timestamped subdirectory."""
    if not os.path.exists(model_dir):
        return None
    
    # Look for VAE_training_* directories
    training_dirs = glob.glob(os.path.join(model_dir, "VAE_training_*"))
    if not training_dirs:
        return None
    
    # Get the most recent training directory
    latest_training_dir = max(training_dirs, key=os.path.getctime)
    final_model_path = os.path.join(latest_training_dir, "final_model")
    
    if os.path.exists(final_model_path):
        return final_model_path
    return None

def generate_embeddings_with_pythae(accession: str, n_dnam_cols: int = 100, latent_dim: int = 512, epochs: int = 20):
    """
    Generate embeddings using Pythae VAE on GSE data.
    
    Args:
        accession (str): GSE accession number (e.g., 'GSE42861')
        n_dnam_cols (int): Number of DNAm columns to use for training
        latent_dim (int): Latent dimension for VAE
        epochs (int): Number of training epochs
    
    Returns:
        str: Path to the generated embeddings CSV file
    """
    _log(f"Starting Pythae embedding generation for {accession}...")
    
    # Create local data directory
    data_dir = "2_poc_simulacra"
    os.makedirs(data_dir, exist_ok=True)
    
    # Define paths - use the actual file names that exist
    embeddings_path = os.path.join(data_dir, f"{accession}_embeddings.csv")
    model_dir = os.path.join(data_dir, f"{accession}_vae_model")
    dnam_path = os.path.join(data_dir, "dnam.csv")
    metadata_path = os.path.join(data_dir, "metadata.csv")
    
    # Check if embeddings already exist
    if os.path.exists(embeddings_path):
        _log(f"Loading existing embeddings from {embeddings_path}...")
        embeddings_df = pd.read_csv(embeddings_path, index_col=0)
        _log(f"Loaded embeddings: {embeddings_df.shape[0]} rows, {embeddings_df.shape[1]} columns")
        return embeddings_path
    
    # Check if model already exists
    model_path = find_model_path(model_dir)
    if model_path:
        _log(f"Loading existing VAE model from {model_path}...")
        model = VAE.load_from_folder(model_path)
        _log("VAE model loaded successfully")
    else:
        _log("Training new VAE model (LONG OPERATION)...")
        train_start = datetime.now()
        
        # Create train/validation split
        ds = {}
        ds['train'], ds['val'] = DNAmDataset.split_dataset(
            split_points=0.8,
            dnam_path=dnam_path,
            metadata_path=metadata_path,
            n_dnam_cols=n_dnam_cols,
            target_metadata_col="disease",
            target_metadata_type="categorical",
            shuffle=True,
            seed=42,
            on_unmatched='warn',
            join_on='Unnamed: 0',
        )
        
        _log(f"Dataset split created - Train: {len(ds['train'])}, Val: {len(ds['val'])}")
        
        # Create VAE model
        model_config = VAEConfig(
            input_dim=(n_dnam_cols,),
            latent_dim=latent_dim
        )
        
        # Create training config
        training_config = BaseTrainerConfig(
            output_dir=model_dir,
            learning_rate=1e-3,
            per_device_train_batch_size=512,
            per_device_eval_batch_size=512,
            num_epochs=epochs,
            keep_best_on_train=True
        )
        
        # Create training pipeline
        pipeline = TrainingPipeline(
            model=VAE(model_config=model_config),
            training_config=training_config
        )
        
        # Train the model
        pipeline(
            train_data=ds['train'],
            eval_data=ds['val']
        )
        
        train_duration = datetime.now() - train_start
        _log(f"VAE training completed in {train_duration.total_seconds():.2f} seconds")
        
        # Find and load the trained model
        model_path = find_model_path(model_dir)
        if model_path:
            model = VAE.load_from_folder(model_path)
            _log("VAE model loaded successfully")
        else:
            raise FileNotFoundError(f"Could not find trained model in {model_dir}")
    
    # Generate embeddings for all data
    _log("Generating embeddings for all data (LONG OPERATION)...")
    embed_start = datetime.now()
    
    # Create dataset for all data
    all_ds = PythaeIterableDataset(
        dnam_path=dnam_path,
        metadata_path=metadata_path,
        n_dnam_cols=n_dnam_cols,
        target_metadata_col="disease",
        target_metadata_type="categorical",
        join_on="Unnamed: 0",
    )
    
    # Get the list of indices from metadata
    index_list = list(all_ds.metadata.index)
    
    # Generate embeddings
    model.eval()
    buffer = []
    with torch.no_grad():
        with open(embeddings_path, 'w') as fout:
            header = ','.join([f'emb_{i}' for i in range(model.latent_dim)])
            fout.write(f'id,{header}\n')
            
            for idx, (X, _) in tqdm(enumerate(all_ds), desc="Generating embeddings"):
                X_tensor = X.to(dtype=torch.float32).unsqueeze(0)
                embedding = model.encoder(X_tensor)['embedding'].cpu().numpy().squeeze()
                
                # Use the original index from metadata
                row_id = index_list[idx] if idx < len(index_list) else idx
                row = ','.join([str(x) for x in embedding])
                buffer.append(f'{row_id},{row}\n')
                
                # Write buffer periodically
                if len(buffer) >= 64:
                    fout.writelines(buffer)
                    buffer.clear()
            
            # Write remaining buffer
            if buffer:
                fout.writelines(buffer)
    
    embed_duration = datetime.now() - embed_start
    _log(f"Embedding generation completed in {embed_duration.total_seconds():.2f} seconds")
    
    # Load and display the embeddings
    embeddings_df = pd.read_csv(embeddings_path, index_col=0)
    _log(f"Generated embeddings: {embeddings_df.shape[0]} rows, {embeddings_df.shape[1]} columns")
    
    return embeddings_path

# Execute the function for GSE42861
embeddings_path = generate_embeddings_with_pythae('GSE42861', n_dnam_cols=100, latent_dim=512, epochs=20)

_log("Pythae embedding generation completed successfully!")
_log(f"Embeddings saved to: {embeddings_path}")

# Display first few rows of embeddings
embeddings_df = pd.read_csv(embeddings_path, index_col=0)
_log(f"Final embeddings shape: {embeddings_df.shape}")
_log("First 5 rows of embeddings:")
display(embeddings_df)


[2025-10-12 21:47:43] Starting Pythae embedding generation for GSE42861...
[2025-10-12 21:47:43] Loading existing VAE model from 2_poc_simulacra\GSE42861_vae_model\VAE_training_2025-10-12_20-48-52\final_model...
[2025-10-12 21:47:44] VAE model loaded successfully
[2025-10-12 21:47:44] Generating embeddings for all data (LONG OPERATION)...


Generating embeddings: 689it [02:00,  5.74it/s] 


[2025-10-12 21:49:44] Embedding generation completed in 120.31 seconds
[2025-10-12 21:49:44] Generated embeddings: 689 rows, 512 columns
[2025-10-12 21:49:44] Pythae embedding generation completed successfully!
[2025-10-12 21:49:44] Embeddings saved to: 2_poc_simulacra\GSE42861_embeddings.csv
[2025-10-12 21:49:44] Final embeddings shape: (689, 512)
[2025-10-12 21:49:44] First 5 rows of embeddings:
               emb_0     emb_1     emb_2     emb_3     emb_4     emb_5  \
id                                                                       
GSM1051525  0.000379  0.007248 -0.011952 -0.006601 -0.002977  0.001650   
GSM1051526 -0.002986  0.003322 -0.002674 -0.010356  0.005399  0.009582   
GSM1051527 -0.006568  0.003980  0.000052 -0.007087 -0.002809  0.004245   
GSM1051528  0.000215  0.009518 -0.009776 -0.007672  0.004302  0.010845   
GSM1051529 -0.001178  0.006284 -0.006167 -0.007700  0.001529  0.008875   

               emb_6     emb_7     emb_8     emb_9  ...   emb_502   emb_503  \
i

In [9]:
# Cell 3: Combine embeddings with target column
import pandas as pd
import numpy as np
from datetime import datetime

def _log(message: str) -> None:
    """Log a message with timestamp."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")

def create_embeddings_with_target(accession: str, target_col: str = "disease"):
    """
    Create a new dataframe combining embeddings with target column.
    
    Args:
        accession (str): GSE accession number (e.g., 'GSE42861')
        target_col (str): Name of the target column from metadata
    
    Returns:
        pd.DataFrame: Combined dataframe with target as first column, followed by embeddings
    """
    _log(f"Creating combined dataframe for {accession} with target column '{target_col}'...")
    
    # Define paths
    data_dir = "2_poc_simulacra"
    embeddings_path = os.path.join(data_dir, f"{accession}_embeddings.csv")
    metadata_path = os.path.join(data_dir, "metadata.csv")
    combined_path = os.path.join(data_dir, f"{accession}_embeddings_with_target.csv")
    
    # Check if combined file already exists
    if os.path.exists(combined_path):
        _log(f"Loading existing combined dataframe from {combined_path}...")
        combined_df = pd.read_csv(combined_path, index_col=0)
        _log(f"Loaded combined dataframe: {combined_df.shape[0]} rows, {combined_df.shape[1]} columns")
        return combined_df
    
    # Load embeddings and metadata
    _log("Loading embeddings and metadata...")
    embeddings_df = pd.read_csv(embeddings_path, index_col=0)
    metadata_df = pd.read_csv(metadata_path, index_col=0)
    
    _log(f"Embeddings shape: {embeddings_df.shape}")
    _log(f"Metadata shape: {metadata_df.shape}")
    
    # Check if target column exists in metadata
    if target_col not in metadata_df.columns:
        available_cols = list(metadata_df.columns)
        raise ValueError(f"Target column '{target_col}' not found in metadata. Available columns: {available_cols}")
    
    # Join on index to combine the dataframes
    _log("Combining embeddings with target column...")
    combined_df = pd.concat([metadata_df[[target_col]], embeddings_df], axis=1, join='inner')
    
    _log(f"Combined dataframe shape: {combined_df.shape}")
    _log(f"Target column '{target_col}' unique values: {combined_df[target_col].unique()}")
    
    # Save the combined dataframe
    _log(f"Saving combined dataframe to {combined_path}...")
    combined_df.to_csv(combined_path)
    _log("Combined dataframe saved successfully")
    
    return combined_df

# Execute the function for GSE42861
combined_df = create_embeddings_with_target('GSE42861', target_col='disease')

_log("Combined dataframe creation completed successfully!")
_log(f"Final combined dataframe shape: {combined_df.shape}")
_log(f"Columns: {list(combined_df.columns)}")
_log("First 5 rows:")
display(combined_df)


[2025-10-12 22:03:27] Creating combined dataframe for GSE42861 with target column 'disease'...
[2025-10-12 22:03:27] Loading embeddings and metadata...
[2025-10-12 22:03:27] Embeddings shape: (689, 512)
[2025-10-12 22:03:27] Metadata shape: (689, 4)
[2025-10-12 22:03:27] Combining embeddings with target column...
[2025-10-12 22:03:27] Combined dataframe shape: (689, 513)
[2025-10-12 22:03:27] Target column 'disease' unique values: ['rheumatoid arthritis' 'Normal']
[2025-10-12 22:03:27] Saving combined dataframe to 2_poc_simulacra\GSE42861_embeddings_with_target.csv...
[2025-10-12 22:03:28] Combined dataframe saved successfully
[2025-10-12 22:03:28] Combined dataframe creation completed successfully!
[2025-10-12 22:03:28] Final combined dataframe shape: (689, 513)
[2025-10-12 22:03:28] Columns: ['disease', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9', 'emb_10', 'emb_11', 'emb_12', 'emb_13', 'emb_14', 'emb_15', 'emb_16', 'emb_17', 'emb_18', 'em

,disease,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_502,emb_503,emb_504,emb_505,emb_506,emb_507,emb_508,emb_509,emb_510,emb_511
GSM1051525,rheumatoid arthritis,0.000379,0.007248,-0.011952,-0.006601,-0.002977,0.001650,-0.002853,-0.006493,-0.003603,...,0.001855,0.004305,0.002511,0.002612,0.007929,-0.006192,-0.000926,-0.000606,0.011051,0.003112
GSM1051526,rheumatoid arthritis,-0.002986,0.003322,-0.002674,-0.010356,0.005399,0.009582,-0.007116,-0.010195,0.004752,...,0.002104,0.006323,0.003349,0.000850,0.008739,-0.000499,-0.003584,-0.003576,0.008540,-0.003069
GSM1051527,rheumatoid arthritis,-0.006568,0.003980,0.000052,-0.007087,-0.002809,0.004245,-0.003250,-0.004742,0.003934,...,0.005058,0.002357,0.002605,0.004585,0.003063,-0.010164,-0.005583,-0.003053,0.012815,0.001405
GSM1051528,rheumatoid arthritis,0.000215,0.009518,-0.009776,-0.007672,0.004302,0.010845,-0.001150,-0.007701,0.006775,...,0.004212,0.011965,0.009021,-0.004224,0.006240,0.002095,0.002774,-0.006467,-0.000980,-0.003478
GSM1051529,rheumatoid arthritis,-0.001178,0.006284,-0.006167,-0.007700,0.001529,0.008875,-0.003372,-0.004588,0.002095,...,0.004174,0.004763,0.007382,0.001549,0.006101,-0.000170,0.001114,-0.002347,0.005288,-0.000686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM1052209,Normal,-0.003765,0.008041,-0.006019,-0.009293,0.003222,0.009334,-0.003292,-0.012899,0.001117,...,0.002203,0.008307,0.006887,-0.001503,0.007390,0.002973,-0.002571,-0.001365,0.000916,0.001138
GSM1052210,Normal,-0.004615,-0.005707,-0.013385,0.008344,-0.020104,0.005249,-0.007903,0.005094,-0.011255,...,-0.016663,-0.014000,-0.005464,-0.007783,0.001713,-0.010014,-0.012169,0.010438,0.003982,0.016364
GSM1052211,Normal,-0.005845,0.007845,-0.008336,-0.003659,0.002398,0.004383,-0.005194,-0.007207,0.000569,...,0.000923,0.008879,0.010149,0.002801,0.009749,-0.000687,-0.004634,-0.000321,0.003844,0.001026
GSM1052212,Normal,-0.005973,0.000146,-0.000831,-0.002002,-0.001332,0.005437,-0.010735,-0.005272,-0.003169,...,0.004573,0.003191,0.005803,0.006951,0.006752,-0.002935,-0.005506,0.006975,0.013059,0.009616


In [1]:
# Cell 4: Benchmark Ridge Classifiers with Different Training Sets
from simulacra.benchmark import (
    _log,
    run_benchmark_experiment,
    save_results_to_csv,
)

# Execute the benchmark experiment with multiple multipliers
all_results, summary_stats = run_benchmark_experiment('GSE42861', seeds=[42, 931782, 8481962], multipliers=(1, 2, 5))

# Save results to CSV
csv_path = save_results_to_csv(all_results, summary_stats, 'GSE42861', target_column='disease')

_log(f"CSV results saved to: {csv_path}")

# Display first few rows of CSV
import pandas as pd
results_df = pd.read_csv(csv_path)
_log(f"\nCSV Preview (first 10 rows):")
display(results_df)


[2025-10-14 15:35:52] Starting benchmark experiment for GSE42861 with seeds: [42, 931782, 8481962] and multipliers: (1.0, 2.0, 5.0)
[2025-10-14 15:35:52] 
=== Running benchmark with seed 42 ===
[2025-10-14 15:35:52] Starting benchmark for GSE42861 with seed 42 and multipliers (1.0, 2.0, 5.0)...
[2025-10-14 15:35:52] Loading combined embeddings with target...
[2025-10-14 15:35:52] Split sizes - Train: 551, Test: 138
[2025-10-14 15:35:52] === Training Baseline Classifier (no augmentation) ===
[2025-10-14 15:35:52] === Training GaussianCopula Augmented Classifiers ===
[2025-10-14 15:35:52] Training GaussianCopula synthesizer (LONG OPERATION)...


C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


[2025-10-14 15:36:28] GaussianCopula synthesizer training completed in 35.87 seconds
[2025-10-14 15:36:28] Generating 5.0x synthetic data with GaussianCopula (LONG OPERATION)...
[2025-10-14 15:36:59] Synthetic data generation completed in 67.64 seconds
[2025-10-14 15:36:59] Testing GaussianCopula with 1.0x augmentation...
[2025-10-14 15:37:00] Testing GaussianCopula with 2.0x augmentation...
[2025-10-14 15:37:00] Testing GaussianCopula with 5.0x augmentation...
[2025-10-14 15:37:00] Total GaussianCopula processing time: 68.37 seconds
[2025-10-14 15:37:00] === Training CTGAN Augmented Classifiers ===
[2025-10-14 15:37:01] Training CTGAN synthesizer (LONG OPERATION)...


C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
disease                2
emb_0                  11
emb_1                  11
emb_2                  11
emb_3                  11
emb_4                  11
emb_5                  11
emb_6                  11
emb_7                  11
emb_8                  11
emb_9                  11
emb_10                 11
emb_11                 11
emb_12                 11
emb_13                 11
emb_14                 11
emb_15                 11
emb_16                 11
emb_17                 11
emb_18                 11
emb_19                 11
emb_20                 11
emb_21                 11
emb_22                 11
emb_23                 11
emb_24                 11
emb_25                 11
emb_26                 11
emb_27                 11
emb_28                 11
emb_29                 11
e

...

C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
disease                2
emb_0                  11
emb_1                  11
emb_2                  11
emb_3                  11
emb_4                  11
emb_5                  11
emb_6                  11
emb_7                  11
emb_8                  11
emb_9                  11
emb_10                 11
emb_11                 11
emb_12                 11
emb_13                 11
emb_14                 11
emb_15                 11
emb_16                 11
emb_17                 11
emb_18                 11
emb_19                 11
emb_20                 11
emb_21                 11
emb_22                 11
emb_23                 11
emb_24                 11
emb_25                 11
emb_26                 11
emb_27                 11
emb_28                 11
emb_29                 11
e

C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


[2025-10-14 19:15:02] TVAE synthesizer training completed in 3173.77 seconds
[2025-10-14 19:15:02] Generating 5.0x synthetic data with TVAE (LONG OPERATION)...
[2025-10-14 19:15:57] Synthetic data generation completed in 3228.67 seconds
[2025-10-14 19:15:57] Testing TVAE with 1.0x augmentation...
[2025-10-14 19:15:58] Testing TVAE with 2.0x augmentation...
[2025-10-14 19:15:58] Testing TVAE with 5.0x augmentation...
[2025-10-14 19:15:59] Total TVAE processing time: 3230.90 seconds
[2025-10-14 19:15:59] Saving benchmark results to 2_poc_simulacra\GSE42861_benchmark_seed_8481962_mults_1-2-5.pkl...
[2025-10-14 19:16:07] Benchmark results saved successfully
[2025-10-14 19:16:07] 
=== Computing Statistics Across Seeds ===
[2025-10-14 19:16:07] 
=== BENCHMARK RESULTS SUMMARY ===
[2025-10-14 19:16:07] Method                Accuracy (mean ± std)    F1 Macro (mean ± std)
[2025-10-14 19:16:07] ----------------------------------------------------------------------
[2025-10-14 19:16:07] baseline  

,accession,dnam_path,metadata_path,target_column,seed,method,multiplier,accuracy,f1_macro,test_size,train_size,timestamp
0,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,baseline,NaN,0.789855,0.789313,138.0,551.0,2025-10-14 19:16:07
1,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,GaussianCopula_1x,1.0,0.811594,0.810959,138.0,1102.0,2025-10-14 19:16:07
2,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,GaussianCopula_2x,2.0,0.811594,0.810959,138.0,1653.0,2025-10-14 19:16:07
3,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,GaussianCopula_5x,5.0,0.804348,0.803512,138.0,3306.0,2025-10-14 19:16:07
4,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,CTGAN_1x,1.0,0.782609,0.782197,138.0,1102.0,2025-10-14 19:16:07
5,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,CTGAN_2x,2.0,0.797101,0.796717,138.0,1653.0,2025-10-14 19:16:07
6,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,CTGAN_5x,5.0,0.789855,0.789756,138.0,3306.0,2025-10-14 19:16:07
7,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,TVAE_1x,1.0,0.782609,0.782563,138.0,1102.0,2025-10-14 19:16:07
8,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,TVAE_2x,2.0,0.797101,0.796931,138.0,1653.0,2025-10-14 19:16:07
9,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,TVAE_5x,5.0,0.760870,0.760857,138.0,3306.0,2025-10-14 19:16:07


In [4]:
%load_ext autoreload
%autoreload 2
# Cell 5: Repeat benchmark with more multipliers: 
# Ridge Classifiers with Different Training Sets
from simulacra.benchmark import (
    _log,
    run_benchmark_experiment,
    save_results_to_csv,
)

# Execute the benchmark experiment with multiple multipliers
all_results, summary_stats = run_benchmark_experiment('GSE42861', seeds=[42, 931782, 8481962], multipliers=(0.2, 0.5, 10))

# Save results to CSV
csv_path = save_results_to_csv(all_results, summary_stats, 'GSE42861', target_column='disease')

_log(f"CSV results saved to: {csv_path}")

# Display first few rows of CSV
import pandas as pd
results_df = pd.read_csv(csv_path)
_log(f"\nCSV Preview (first 10 rows):")
display(results_df)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[2025-10-14 23:44:52] Starting benchmark experiment for GSE42861 with seeds: [42, 931782, 8481962] and multipliers: (0.2, 0.5, 10.0)
[2025-10-14 23:44:52] 
=== Running benchmark with seed 42 ===
[2025-10-14 23:44:52] Starting benchmark for GSE42861 with seed 42 and multipliers (0.2, 0.5, 10.0)...
[2025-10-14 23:44:52] CUDA usage: Enabled
[2025-10-14 23:44:52] Loading combined embeddings with target...
[2025-10-14 23:44:52] Split sizes - Train: 551, Test: 138
[2025-10-14 23:44:52] === Training Baseline Classifier (no augmentation) ===
[2025-10-14 23:44:53] === Training GaussianCopula Augmented Classifiers ===
[2025-10-14 23:44:53] Creating new synthesizer for GaussianCopula...
[2025-10-14 23:44:53] Creating metadata for GaussianCopula...
[2025-10-14 23:44:53] Metadata saved to 2_poc_simulacra\GSE42861_GaussianCopula_metadata_seed_42.json
[2025-10-14 23:44:53] Training GaussianCopula synthesizer (LONG

C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


[2025-10-14 23:47:33] Saving trained synthesizer to 2_poc_simulacra\GSE42861_GaussianCopula_synthesizer_seed_42.pkl
[2025-10-14 23:47:34] Synthesizer saved successfully
[2025-10-14 23:47:34] GaussianCopula synthesizer training completed in 161.41 seconds
[2025-10-14 23:47:34] Generating 10.0x synthetic data with GaussianCopula (LONG OPERATION)...
[2025-10-14 23:48:27] Synthetic data generation completed in 214.14 seconds
[2025-10-14 23:48:27] Testing GaussianCopula with 0.2x augmentation...
[2025-10-14 23:48:27] Testing GaussianCopula with 0.5x augmentation...
[2025-10-14 23:48:27] Testing GaussianCopula with 10.0x augmentation...
[2025-10-14 23:48:28] Total GaussianCopula processing time: 215.26 seconds
[2025-10-14 23:48:28] === Training CTGAN Augmented Classifiers ===
[2025-10-14 23:48:28] Creating new synthesizer for CTGAN...
[2025-10-14 23:48:28] Creating metadata for CTGAN...
[2025-10-14 23:48:29] Metadata saved to 2_poc_simulacra\GSE42861_CTGAN_metadata_seed_42.json
[2025-10-14 2

C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
disease                2
emb_0                  11
emb_1                  11
emb_2                  11
emb_3                  11
emb_4                  11
emb_5                  11
emb_6                  11
emb_7                  11
emb_8                  11
emb_9                  11
emb_10                 11
emb_11                 11
emb_12                 11
emb_13                 11
emb_14                 11
emb_15                 11
emb_16                 11
emb_17                 11
emb_18                 11
emb_19                 11
emb_20                 11
emb_21                 11
emb_22                 11
emb_23                 11
emb_24                 11
emb_25                 11
emb_26                 11
emb_27                 11
emb_28                 11
emb_29                 11
e

...

C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
disease                2
emb_0                  11
emb_1                  11
emb_2                  11
emb_3                  11
emb_4                  11
emb_5                  11
emb_6                  11
emb_7                  11
emb_8                  11
emb_9                  11
emb_10                 11
emb_11                 11
emb_12                 11
emb_13                 11
emb_14                 11
emb_15                 11
emb_16                 11
emb_17                 11
emb_18                 11
emb_19                 11
emb_20                 11
emb_21                 11
emb_22                 11
emb_23                 11
emb_24                 11
emb_25                 11
emb_26                 11
emb_27                 11
emb_28                 11
emb_29                 11
e

C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:162: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\ferdi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdv\single_table\base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


[2025-10-15 02:03:13] Saving trained synthesizer to 2_poc_simulacra\GSE42861_TVAE_synthesizer_seed_8481962.pkl
[2025-10-15 02:03:14] Synthesizer saved successfully
[2025-10-15 02:03:14] TVAE synthesizer training completed in 1287.35 seconds
[2025-10-15 02:03:14] Generating 10.0x synthetic data with TVAE (LONG OPERATION)...
[2025-10-15 02:03:49] Synthetic data generation completed in 1321.95 seconds
[2025-10-15 02:03:49] Testing TVAE with 0.2x augmentation...
[2025-10-15 02:03:49] Testing TVAE with 0.5x augmentation...
[2025-10-15 02:03:49] Testing TVAE with 10.0x augmentation...
[2025-10-15 02:03:49] Total TVAE processing time: 1322.62 seconds
[2025-10-15 02:03:49] Saving benchmark results to 2_poc_simulacra\GSE42861_benchmark_seed_8481962_mults_0.2-0.5-10.pkl...
[2025-10-15 02:03:52] Benchmark results saved successfully
[2025-10-15 02:03:52] 
=== Computing Statistics Across Seeds ===
[2025-10-15 02:03:52] 
=== BENCHMARK RESULTS SUMMARY ===
[2025-10-15 02:03:52] Method                A

,accession,dnam_path,metadata_path,target_column,seed,method,multiplier,accuracy,f1_macro,test_size,train_size,timestamp
0,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,baseline,NaN,0.789855,0.789313,138.0,551.0,2025-10-15 02:03:52
1,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,GaussianCopula_0.2x,0.2,0.782609,0.782197,138.0,661.0,2025-10-15 02:03:52
2,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,GaussianCopula_0.5x,0.5,0.789855,0.789579,138.0,826.0,2025-10-15 02:03:52
3,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,GaussianCopula_10x,10.0,0.804348,0.803512,138.0,6061.0,2025-10-15 02:03:52
4,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,CTGAN_0.2x,0.2,0.789855,0.789313,138.0,661.0,2025-10-15 02:03:52
5,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,CTGAN_0.5x,0.5,0.789855,0.789313,138.0,826.0,2025-10-15 02:03:52
6,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,CTGAN_10x,10.0,0.724638,0.724116,138.0,6061.0,2025-10-15 02:03:52
7,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,TVAE_0.2x,0.2,0.789855,0.789313,138.0,661.0,2025-10-15 02:03:52
8,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,TVAE_0.5x,0.5,0.789855,0.789313,138.0,826.0,2025-10-15 02:03:52
9,GSE42861,2_poc_simulacra\dnam.csv,2_poc_simulacra\metadata.csv,disease,42,TVAE_10x,10.0,0.739130,0.739076,138.0,6061.0,2025-10-15 02:03:52
